# 🎬 ClipNest — Google Colab Deploy

Yeh notebook **ClipNest** (self-hosted video downloader) ko Colab par chalata hai aur ek public
`https://....trycloudflare.com` link deta hai.

**Chalane ka tarika:** upar menu se **Runtime → Run all**, ya cells ek-ek karke chalayein.
Pehli baar lagbhag **2–3 minute** lagte hain.

> ⚠️ **Zaroori baatein**
> - Colab **temporary** hai — tab band hone ya ~12 ghante ke baad server aur downloaded files delete ho jaate hain.
> - Public link par koi login nahi hota — jise bhi link dein wo use kar sakta hai. Sensitive use ke liye VPS/Docker better hai.
> - Colab ke terms follow karein — ise proxy, mining ya abusive traffic ke liye use na karein.
> - Sirf wahi content download karein jiska aapko haq hai.


## 1️⃣ Install (clone + dependencies)


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/gintokisakata918-prog/clipnest.git"
APP_DIR = Path("/content/clipnest")


def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, cwd=cwd, check=check)


# ffmpeg (quality merge / MP3 ke liye)
if subprocess.run("command -v ffmpeg", shell=True, capture_output=True).returncode != 0:
    run("apt-get -qq update && apt-get -qq install -y ffmpeg")

# code
if APP_DIR.exists():
    run("git fetch --depth 1 origin main && git reset --hard origin/main", cwd=APP_DIR)
else:
    run(f"git clone --depth 1 {REPO} {APP_DIR}")

# dependencies (Colab ke preinstalled packages se conflict na ho, isliye repo install nahi kar rahe)
run(
    f"{sys.executable} -m pip install -q --upgrade "
    '"fastapi>=0.115" "uvicorn[standard]>=0.32" "yt-dlp>=2025.1.15" '
    '"httpx>=0.27" "pydantic>=2.9" "pydantic-settings>=2.6"'
)

(APP_DIR / "downloads").mkdir(exist_ok=True)
(APP_DIR / "data").mkdir(exist_ok=True)
print("\n✅ Install ho gaya —", APP_DIR)

## 2️⃣ Settings (optional)

Apni zaroorat ke hisaab se values badal dein, phir agli cell chalayein.


In [ ]:
import os

os.environ["CLIPNEST_DOWNLOAD_DIR"] = "/content/clipnest/downloads"
os.environ["CLIPNEST_MAX_CONCURRENT_DOWNLOADS"] = "2"  # ek saath downloads
os.environ["CLIPNEST_JOB_TTL_MINUTES"] = "30"  # file kitni der rahegi
os.environ["CLIPNEST_MAX_FILESIZE_MB"] = "1024"  # per-file limit
os.environ["CLIPNEST_MAX_DURATION_SECONDS"] = "7200"  # 2 ghante
os.environ["CLIPNEST_RATE_LIMIT_PER_10MIN"] = "30"  # per-IP requests

print("Settings set ho gayi ✅")

## 3️⃣ Cookies (optional)

Private / age-restricted content ke liye apni `cookies.txt` (Netscape format) upload karein.
Yeh cell **skip** kar sakte hain — public videos iske bina bhi chalte hain.


In [ ]:
from pathlib import Path

try:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        name = next(iter(uploaded))
        target = Path("/content/clipnest/data/cookies.txt")
        target.write_bytes(uploaded[name])
        import os

        os.environ["CLIPNEST_COOKIES_FILE"] = str(target)
        print(f"Cookies save ho gayi: {target}")
        print("Ab neeche wala server cell chalayein.")
    else:
        print("Koi file upload nahi hui — skip.")
except ImportError:
    print("Yeh cell sirf Google Colab ke liye hai — skip kar dein.")

## 4️⃣ Server start karein + public link


In [ ]:
import os
import re
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

APP_DIR = Path("/content/clipnest")
PORT = 8000
SERVER_LOG = Path("/content/clipnest-server.log")
TUNNEL_LOG = Path("/content/clipnest-tunnel.log")

# purane process band karein (dobara chalane par kaam aata hai)
for pattern in ("uvicorn app.main:app", "cloudflared tunnel"):
    subprocess.run(f"pkill -f '{pattern}'", shell=True, check=False)

time.sleep(1)
SERVER_LOG.write_text("")

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=APP_DIR,
    stdout=SERVER_LOG.open("w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
    env=os.environ.copy(),
)

for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    print(SERVER_LOG.read_text()[-3000:])
    raise RuntimeError("Server start nahi hua — upar ka log dekhein.")

print(f"✅ Server chal raha hai: http://127.0.0.1:{PORT}")

# cloudflared (free public tunnel, koi account nahi chahiye)
binary = Path("/usr/local/bin/cloudflared")
if not binary.exists():
    subprocess.run(
        "wget -q -O /usr/local/bin/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 "
        "&& chmod +x /usr/local/bin/cloudflared",
        shell=True,
        check=True,
    )

TUNNEL_LOG.write_text("")
subprocess.Popen(
    [str(binary), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=TUNNEL_LOG.open("w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

public_url = None
for _ in range(120):
    match = re.search(
        r"https://[a-z0-9-]+\.trycloudflare\.com", TUNNEL_LOG.read_text(errors="ignore")
    )
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if not public_url:
    print(TUNNEL_LOG.read_text(errors="ignore")[-2000:])
    raise RuntimeError("Tunnel URL nahi mila — yeh cell dobara chalayein.")

print("=" * 62)
print("🎬 ClipNest live hai!")
print("🔗 App   :", public_url)
print("🩺 Health:", public_url + "/api/health")
print("📚 Docs  :", public_url + "/api/docs")
print("=" * 62)
print("Link kisi ke saath bhi share kar sakte hain (koi login nahi).")
print("Server tab tak chalega jab tak yeh Colab session active hai.")

## 5️⃣ Check karein ki public link kaam kar raha hai


In [ ]:
import json
import urllib.request

url = globals().get("public_url")
if not url:
    raise RuntimeError("Pehle upar wala server cell chalayein.")

with urllib.request.urlopen(url + "/api/health", timeout=30) as response:
    print(json.dumps(json.load(response), indent=2))

print("\nSab theek hai ✅ — ab link browser mein kholiye.")

## 🛠️ Zaroori commands

**Server restart karein** (settings ya cookies badalne ke baad): cell **4** dobara chalayein.

**Naya code pull karein:** cell **1** dobara chalayein, phir cell **4**.

**Server band karein:** neeche wala cell chalayein.

> 💡 Colab free session ~12 ghante ya disconnect hone tak chalta hai. Permanent hosting ke liye
> isi repo ko kisi VPS par `docker compose up -d` se chalayein.


In [ ]:
import subprocess

subprocess.run("pkill -f 'uvicorn app.main:app'", shell=True, check=False)
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True, check=False)
print("🛑 Server aur tunnel band ho gaye.")